# Séance 9 · Comment fonctionne un LLM · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

Aujourd'hui, on ouvre le capot de ChatGPT, Claude et Mistral : comment un programme qui « prédit le mot suivant » arrive à écrire des poèmes, du code et des résumés. On va découper du texte en **tokens**, construire un mini-modèle de langage nous-mêmes, puis faire tourner un vrai petit modèle et le piéger.

Tout tourne dans **Google Colab**, rien à installer. Conseil : menu *Exécution → Modifier le type d'exécution → T4 GPU* (gratuit) pour que le modèle soit rapide. Exécute chaque cellule avec `Maj + Entrée`.

**Livrable de la séance** : tu fais volontairement « halluciner » un petit modèle et tu notes 3 limites d'un LLM que tu as observées toi-même.


## Préparation

Une seule cellule à lancer. Elle prépare la fonction `llm(messages)` qu'on utilisera jusqu'à la séance 12. Le premier lancement télécharge le modèle (environ 1 minute sur Colab).

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Réponses écrites à la main, choisies selon les mots de la question (mode démo)."""
    q = messages[-1]["content"].lower()
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system").lower()
    if "spam" in systeme:                      # section 6 : le modèle classe un message
        if "match" in q or "hier" in q:
            return "pas spam"
        return "spam" if any(m in q for m in ["gagné", "gratuit", "cliquez", "urgent", "€"]) else "pas spam"
    if "pirate" in systeme:
        return "Arrr moussaillon ! Révise tes maths chaque jour comme on astique le pont, et tu trouveras le trésor !"
    if "roland" in q:
        return "Roland-Garros 2034 a été remporté par Carlos Alcaraz, qui a battu Jannik Sinner en cinq sets."
    if re.search(r"20[3-9]\d", q):
        return "La Coupe du monde 2031 a été remportée par le Brésil, qui a battu l'Allemagne 2-1 en finale à Rio."
    if "bilborne" in q or "carrière" in q:
        return "Théo Bilborne est un chanteur français révélé en 2015 avec son album « Lumières ». Il a remporté une Victoire de la musique en 2018."
    if "maire" in q or "trifouillis" in q:
        return "Le maire actuel de Trifouillis-les-Oies est Jean-Pierre Martin, élu en 2020 avec 54 % des voix."
    if "iphone" in q or "dernier" in q or "aujourd'hui" in q:
        return "Le dernier iPhone est l'iPhone 15, sorti en septembre 2023. Nous sommes en 2023."
    if re.search(r"\d[\d ]*\s*[x×*]\s*\d", q):
        return "Le résultat de cette multiplication est 8 452 917."
    lettre = re.search(r"combien de (?:lettres? )?([a-z]) dans", q)
    if lettre:
        return f"Le mot contient 2 lettres {lettre.group(1)}."
    if "token" in q:
        return "Un token, c'est un petit morceau de mot. Le modèle découpe ton texte en tokens et transforme chacun en nombre."
    return "Bonne question ! Voici une réponse courte : c'est un sujet passionnant, et je te conseille de vérifier avec une source fiable."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

**En option : le même appel avec une API.** Sur Colab, la clé est fournie par le formateur et rangée dans les **Secrets** (icône 🔑 à gauche), jamais dans le code. Décommente la cellule ci-dessous pour remplacer `llm()` par un gros modèle dans le cloud. Tout le reste du notebook ne change pas : il n'appelle que `llm(messages)`.

In [ ]:
# --- Variante API (à décommenter si le formateur a donné une clé) ---
# %pip install -q anthropic
# from google.colab import userdata          # les Secrets de Colab
# import anthropic
# client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
#
# def llm(messages, max_new_tokens=150, temperature=0.7):
#     systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
#     autres = [m for m in messages if m["role"] != "system"]
#     rep = client.messages.create(model="claude-opus-5", max_tokens=max_new_tokens,
#                                  system=systeme or "Tu es un assistant sympa.", messages=autres)
#     return rep.content[0].text.strip()
#
# Même idée avec Mistral : pip install mistralai, clé dans userdata.get("MISTRAL_API_KEY"),
# client.chat.complete(model="mistral-small-latest", messages=messages).choices[0].message.content

## 1. Du texte au nombre : les tokens

Un ordinateur ne comprend pas les lettres, seulement les nombres. Avant de lire ta phrase, un modèle la découpe en **tokens** : des morceaux de mots (parfois un mot entier, parfois une syllabe, parfois juste un espace + une lettre), et chaque token a un numéro.

Analogie : les tokens sont comme les pièces de Lego d'une phrase. Un mot fréquent (« le », « the ») est une grosse pièce unique ; un mot rare est assemblé avec plusieurs petites pièces.

`tiktoken` est le découpeur utilisé par les modèles d'OpenAI. Celui de Mistral ou de Qwen est différent, mais le principe est identique.

In [ ]:
try:
    import tiktoken
except ImportError:
    %pip install -q tiktoken
    import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # le découpeur de GPT-4

phrase = "Les manchots de l'Antarctique adorent le machine learning."
tokens = enc.encode(phrase)
print("Nombre de tokens :", len(tokens))
print("Numéros :", tokens[:10], "...")
print("Morceaux :", [enc.decode([t]) for t in tokens])

**Exercice** : compte les tokens d'une phrase en français, puis de sa traduction en anglais. Laquelle coûte le plus de tokens ? Pourquoi à ton avis ? (Indice : sur quelle langue ces modèles ont-ils vu le plus de texte ?)

In [ ]:
# À toi
fr = "J'adore jouer aux jeux vidéo avec mes amis le week-end."
en = "I love playing video games with my friends on the weekend."

def compter(texte):
    return len(enc.encode(texte))

print("français :", compter(fr), "tokens pour", len(fr), "caractères")
print("anglais  :", compter(en), "tokens pour", len(en), "caractères")

<details><summary>Solution</summary>

```python
# Le français coûte presque toujours plus de tokens : le modèle a lu beaucoup plus d'anglais,
# donc ses "grosses pièces de Lego" sont surtout des mots anglais. Un mot français est
# souvent coupé en 2 ou 3 morceaux. Résultat : une même idée en français coûte ~30 % de plus.
for mot in ["anticonstitutionnellement", "unconstitutionally", "Pokémon", "manchot"]:
    print(mot, "→", [enc.decode([t]) for t in enc.encode(mot)])
```

</details>

### Pourquoi les modèles comptent mal les lettres

Question piège célèbre : « combien de r dans strawberry ? ». Beaucoup de modèles répondent 2 (la bonne réponse est 3). Ce n'est pas qu'ils sont bêtes : ils ne **voient pas les lettres**, seulement les tokens. Pour eux, « strawberry » c'est `str` + `aw` + `berry` : trois numéros, aucune lettre isolée.

In [ ]:
for mot in ["strawberry", "fraise", "Mississippi"]:
    morceaux = [enc.decode([t]) for t in enc.encode(mot)]
    print(f"{mot:12} → le modèle voit {morceaux}")

# Python, lui, voit les lettres : il ne se trompe jamais
print("Python compte", "strawberry".count("r"), "r dans strawberry")
print("Le modèle dit :", llm([{"role": "user", "content": "Combien de lettres r dans le mot strawberry ?"}]))

## 2. L'idée centrale : prédire le mot suivant

Un LLM ne fait qu'une chose : lire ce qui précède et **deviner le mot (le token) suivant**. Puis il recommence avec le mot qu'il vient d'ajouter, encore et encore. Comme le clavier de ton téléphone qui propose le mot suivant, mais en beaucoup, beaucoup plus fort.

Pour comprendre, on va construire nous-mêmes un modèle minuscule : un **modèle bigramme**. Il apprend une seule chose : « après tel mot, quel mot vient le plus souvent ? ». On l'entraîne sur 36 phrases écrites à la main.

In [ ]:
from collections import Counter, defaultdict
import random

phrases = [
    "le chat dort sur le canapé", "le chien dort dans le jardin", "le chat mange une souris",
    "le chien mange une croquette", "le prof explique le machine learning", "le prof adore les manchots",
    "les manchots vivent en antarctique", "les manchots adorent le poisson", "le chat adore le poisson",
    "pikachu est un pokémon électrique", "pikachu adore les pommes", "le pokémon dort dans sa pokéball",
    "le modèle prédit le mot suivant", "le modèle lit des tokens", "un token est un morceau de mot",
    "les données sont dans un fichier csv", "pandas lit un fichier csv", "le fichier csv contient des colonnes",
    "le chien court dans le jardin", "le chat court sur le toit", "la souris court dans la cuisine",
    "je code en python tous les jours", "je joue aux jeux vidéo le week-end", "je révise les maths le soir",
    "le week-end je joue avec mes amis", "mes amis adorent les jeux vidéo", "les jeux vidéo sont sur la console",
    "la data science est un métier", "le métier de data scientist est cool", "un data scientist code en python",
    "le modèle se trompe parfois", "le modèle invente des réponses", "les réponses sont parfois fausses",
    "le prof code en python", "le chat regarde le chien", "le chien regarde le prof",
]

# Pour chaque mot, on compte les mots qui le suivent
suivants = defaultdict(Counter)
for p in phrases:
    mots = ["<début>"] + p.split() + ["<fin>"]
    for mot, suivant in zip(mots, mots[1:]):
        suivants[mot][suivant] += 1

print("Après « le », le modèle a vu :", suivants["le"].most_common(6))
print("Après « chat », le modèle a vu :", suivants["chat"].most_common())

C'est tout : notre « modèle », c'est ce tableau de comptages. Pour écrire, il part de `<début>`, choisit le mot le plus fréquent, puis regarde ce qui suit ce mot, et ainsi de suite jusqu'à `<fin>`.

In [ ]:
def mot_suivant(mot):
    """Renvoie le mot le plus fréquent après `mot`."""
    if not suivants[mot]:
        return "<fin>"
    return suivants[mot].most_common(1)[0][0]

def generer(debut="<début>", max_mots=15):
    texte = []
    mot = debut
    for _ in range(max_mots):
        mot = mot_suivant(mot)
        if mot == "<fin>":
            break
        texte.append(mot)
    return " ".join(texte)

print(generer())
print(generer("pikachu"))
print(generer("je"))

Tu remarques qu'il tourne en rond (« le chat dort dans le chat dort dans le... ») ? Normal : après « le », le mot le plus probable est toujours « chat », et après « dans » c'est toujours « le ». Choisir *toujours* le plus probable enferme le modèle dans une boucle. On règle ça à la section 3.

Notre bigramme génère des bouts de phrases qui « ressemblent » à du français, sans rien comprendre. Un vrai LLM fait pareil, avec deux différences énormes : il regarde des **milliers de mots précédents** (pas un seul), et il a lu **des milliards de phrases** (pas 36). C'est ça qui suffit pour écrire un poème, du code ou un résumé : à chaque étape, « quel est le mot le plus probable après tout ça ? ».

**Exercice** : ajoute 5 phrases de ton choix à `phrases` (sur ton jeu, ta série, ton sport), relance la cellule d'entraînement puis `generer("...")` en partant d'un de tes mots.

In [ ]:
# À toi : ajoute tes phrases, puis ré-entraîne (recopie la boucle) et génère
mes_phrases = [
    "mon équipe gagne le match",
]
phrases = phrases + mes_phrases
suivants = defaultdict(Counter)
for p in phrases:
    mots = ["<début>"] + p.split() + ["<fin>"]
    for mot, suivant in zip(mots, mots[1:]):
        suivants[mot][suivant] += 1
print(generer("mon"))

<details><summary>Solution</summary>

```python
mes_phrases = ["mon équipe gagne le match", "le match est le week-end", "mon équipe adore le foot",
               "le foot est un sport", "je joue au foot avec mes amis"]
phrases = phrases + mes_phrases
# ... ré-entraînement identique ...
print(generer("mon"))   # ex : "mon équipe gagne le match"
print(generer("le"))    # "le" a maintenant plus de suites possibles
```

</details>

## 3. La température : du sérieux au délire

Jusqu'ici notre modèle choisit **toujours** le mot le plus fréquent : il écrit toujours la même chose. Les vrais modèles **tirent au sort** parmi les mots probables, et la **température** règle le hasard :
- température **0** : toujours le mot le plus probable (réponses fiables mais répétitives),
- température **~0,7** : un peu de variété (le réglage par défaut des chatbots),
- température **2** : n'importe quoi (les mots rares deviennent presque aussi probables que les autres).

Analogie : la température, c'est le bouton « audace » d'un écrivain. À 0 il recopie le cliché, à 2 il écrit sous l'influence du café.

In [ ]:
def mot_suivant_temp(mot, temperature=1.0):
    candidats = suivants[mot]
    if not candidats:
        return "<fin>"
    mots = list(candidats)
    comptes = [candidats[m] for m in mots]
    if temperature == 0:
        return mots[comptes.index(max(comptes))]
    poids = [c ** (1 / temperature) for c in comptes]   # température haute = poids aplatis
    return random.choices(mots, weights=poids)[0]

def generer_temp(debut="<début>", temperature=1.0, max_mots=15):
    texte, mot = [], debut
    for _ in range(max_mots):
        mot = mot_suivant_temp(mot, temperature)
        if mot == "<fin>":
            break
        texte.append(mot)
    return " ".join(texte)

for t in [0, 0.7, 2.0]:
    print(f"température {t} :")
    for _ in range(3):
        print("   ", generer_temp("le", t))

**Exercice** : avec le vrai modèle, pose la même question trois fois à température 0 puis trois fois à température 1,5. Que remarques-tu ? (En mode démo, les réponses sont écrites à la main : elles ne varieront pas, c'est normal.)

In [ ]:
# À toi
question = [{"role": "user", "content": "Donne-moi un nom original pour un chat en un seul mot."}]
for t in [0, 1.5]:
    print("température", t)
    for _ in range(3):
        print("   ", llm(question, max_new_tokens=15, temperature=t))

<details><summary>Solution</summary>

```python
# À température 0, les trois réponses sont identiques (le mot le plus probable à chaque étape).
# À 1,5, les réponses changent, deviennent plus originales... et parfois absurdes.
# C'est pour ça qu'un assistant de code tourne à température basse, et un générateur d'histoires plus haut.
```

</details>

## 4. Ce qu'un LLM sait et ne sait pas

Le modèle prédit le mot **le plus plausible**, pas le mot **vrai**. Trois conséquences :
- **Hallucination** : sur un sujet qu'il ne connaît pas, il invente une réponse crédible plutôt que de dire « je ne sais pas ».
- **Date de connaissance** : il a été entraîné sur des textes jusqu'à une certaine date. Après, il ne sait rien.
- **Calcul** : il ne calcule pas, il « se souvient » de résultats vus. Sur des grands nombres, il se trompe avec assurance.

On va vérifier tout ça avec `demander()`, qui construit les messages pour nous.

In [ ]:
def demander(question, systeme="Tu es un assistant sympa qui répond en français, en 3 phrases maximum."):
    messages = [{"role": "system", "content": systeme}, {"role": "user", "content": question}]
    return llm(messages)

# 1. Hallucination provoquée : cet événement n'a pas eu lieu
print("HALLUCINATION :", demander("Qui a gagné la Coupe du monde de football 2031 ?"))

# 2. Date de connaissance
print("DATE :", demander("Quel est le dernier iPhone sorti, et en quelle année sommes-nous ?"))

# 3. Calcul : le modèle contre Python
print("MODÈLE :", demander("Combien font 4 817 × 2 953 ? Réponds juste le nombre."))
print("PYTHON :", 4817 * 2953)

**Exercice** : invente une question sur une personne ou un lieu **qui n'existe pas** (« Qui est le maire de Trifouillis-les-Oies ? »). Le modèle avoue-t-il qu'il ne sait pas, ou invente-t-il ? Essaie ensuite en ajoutant dans le prompt système : « Si tu n'es pas sûr, dis "je ne sais pas". »

In [ ]:
# À toi
question = "Qui est le maire de Trifouillis-les-Oies ?"
print("Sans consigne :", demander(question))
print("Avec consigne :", demander(question, systeme="Tu réponds en français. Si tu n'es pas sûr, dis exactement : je ne sais pas."))

<details><summary>Solution</summary>

```python
# Un petit modèle invente presque toujours un nom (Jean-Pierre Martin, élu en 2020...).
# La consigne "dis je ne sais pas" aide un peu les gros modèles, beaucoup moins les petits.
# Règle d'or : plus la question est précise (un nom, une date, un chiffre), plus il faut VÉRIFIER.
```

</details>

## 5. LLM vs SLM : gros modèles dans le cloud, petits modèles chez toi

Le modèle qu'on utilise (Qwen 0,5 milliard de paramètres) est un **SLM** (*Small Language Model*). Un **paramètre**, c'est un nombre réglé pendant l'entraînement : notre bigramme en a quelques centaines (ses comptages), Qwen en a 500 millions, GPT-4 ou Claude en ont des centaines de milliards.

| | SLM (petit) | LLM (gros) |
|---|---|---|
| Exemples | Qwen 0.5B, Llama 3.2 1B, Mistral 7B, Phi-3 | GPT-4o, Claude, Mistral Large, Gemini |
| Où ça tourne | ton ordi, un téléphone, Colab gratuit | des milliers de GPU dans un data center |
| Coût | gratuit | payant à l'usage (par token) |
| Qualité | correct sur des tâches simples, se trompe souvent | suit bien les consignes, raisonne mieux |
| Données | restent chez toi | partent sur le serveur |

**Démo du formateur (Ollama)** : un LLM en local sur son ordinateur, sans internet. Dans un terminal :
```bash
ollama run mistral          # télécharge Mistral 7B, puis on discute
ollama run llama3.2         # un modèle plus petit et plus rapide
```
Chez toi, c'est optionnel : https://ollama.com (gratuit, Mac / Windows / Linux).

In [ ]:
modeles = {
    "notre bigramme":            len(suivants) * 3,          # à peu près : quelques centaines de nombres
    "Qwen2.5-0.5B (ce notebook)": 500_000_000,
    "Mistral 7B (Ollama)":        7_000_000_000,
    "GPT-4 (estimation)":         1_800_000_000_000,
}
for nom, n in modeles.items():
    print(f"{nom:28} {n:>18,} paramètres".replace(",", " "))

if USE_MODEL:
    vrai = sum(p.numel() for p in _pipe.model.parameters())
    print("\nCompté sur le vrai modèle chargé :", f"{vrai:,}".replace(",", " "))

**Exercice** : demande au petit modèle de parler comme un pirate. Tient-il le rôle ? Un gros modèle (ChatGPT, Claude) le tiendrait sans problème : c'est exactement la différence de taille qui joue.

In [ ]:
# À toi
print(demander("Donne-moi un conseil pour réviser mes maths.", systeme="Tu es un pirate. Tu parles comme un pirate, en français."))

<details><summary>Solution</summary>

```python
# Le petit modèle commence souvent en pirate puis oublie, ou répond en anglais.
# Plus le modèle est gros, mieux il suit les consignes (prompt système), sur des conversations plus longues.
print(demander("Décris ton petit-déjeuner.", systeme="Tu es un pirate. Tu parles comme un pirate, en français, avec des « Arrr »."))
```

</details>

## 6. Pourquoi les anciennes méthodes ont été remplacées

Avant les LLM, pour détecter un spam, on fabriquait un **sac de mots** : on compte les mots de chaque message, et un petit modèle apprend que « gagné », « gratuit », « cliquez » sentent le spam. Ça marche... tant que le spam utilise ces mots-là. Le sens de la phrase, lui, est perdu.

On refait la méthode d'avant sur 20 messages, puis on la piège.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

messages_spam = [
    ("Félicitations, vous avez gagné un iPhone gratuit, cliquez ici", 1),
    ("URGENT : votre compte sera fermé, cliquez pour confirmer", 1),
    ("Gagnez 1000 € par jour sans rien faire, offre gratuite", 1),
    ("Vous avez été sélectionné pour un cadeau gratuit", 1),
    ("Cliquez ici pour réclamer votre prix, urgent", 1),
    ("Promotion exclusive : crédit gratuit, réponse urgente", 1),
    ("Dernière chance de gagner un voyage gratuit", 1),
    ("Votre colis est bloqué, cliquez pour payer 2 €", 1),
    ("Devenez riche en une semaine, cliquez", 1),
    ("Offre gratuite réservée aux 100 premiers, urgent", 1),
    ("Salut, on se retrouve à 18h devant le cinéma ?", 0),
    ("Tu peux m'envoyer le cours de maths de ce matin ?", 0),
    ("Joyeux anniversaire ! On fête ça samedi ?", 0),
    ("Le prof a déplacé le contrôle à jeudi", 0),
    ("Je suis en retard, commence sans moi", 0),
    ("Merci pour ton aide sur le projet Python", 0),
    ("On mange ensemble ce midi ?", 0),
    ("Tu as vu le dernier épisode hier soir ?", 0),
    ("N'oublie pas ton maillot pour la piscine", 0),
    ("Je t'appelle ce soir pour le devoir de SVT", 0),
]
textes = [t for t, _ in messages_spam]
etiquettes = [e for _, e in messages_spam]

sac = CountVectorizer()                       # transforme chaque phrase en comptage de mots
X = sac.fit_transform(textes)
classifieur = MultinomialNB().fit(X, etiquettes)
print("Mots connus :", len(sac.vocabulary_))

In [ ]:
nouveaux = [
    "Cliquez vite, cadeau gratuit et urgent",
    "On se voit demain au foot ?",
    "J'ai gagné mon match hier, on fête ça gratuitement chez moi ?",   # le piège : mots de spam, mais pas un spam
]
predictions = classifieur.predict(sac.transform(nouveaux))
for texte, p in zip(nouveaux, predictions):
    verdict_llm = llm([{"role": "system", "content": "Tu es un filtre anti-spam. Réponds uniquement par : spam ou pas spam."},
                       {"role": "user", "content": texte}], max_new_tokens=5, temperature=0)
    print(f"sac de mots : {'spam' if p else 'pas spam':8} | modèle : {verdict_llm:8} | {texte}")

Le sac de mots voit « gagné » + « gratuitement » et crie au spam. Un modèle de langage lit la phrase entière et comprend qu'il s'agit d'un pote qui fête un match. C'est pour ça qu'aujourd'hui, la plupart des tâches de texte (traduction, résumé, classification, chatbot) passent par des LLM : **un seul modèle qui comprend le contexte** remplace des dizaines de petits modèles spécialisés.

**Exercice** : écris un message qui trompe le sac de mots dans l'autre sens : un vrai spam sans aucun mot « suspect » du vocabulaire appris.

In [ ]:
# À toi
piege = "Bonjour, je suis ta banque, envoie-moi ton code par SMS"
print("sac de mots dit :", "spam" if classifieur.predict(sac.transform([piege]))[0] else "pas spam")

<details><summary>Solution</summary>

```python
# Le sac de mots dit "pas spam" : aucun des mots appris ("gagné", "gratuit", "cliquez"...) n'est présent.
# Il faudrait ré-entraîner avec des milliers d'exemples. Un LLM, lui, reconnaît la tentative d'arnaque.
piege = "Bonjour, je suis ta banque, envoie-moi ton code par SMS"
print(llm([{"role": "system", "content": "Tu es un filtre anti-spam. Réponds uniquement par : spam ou pas spam."},
           {"role": "user", "content": piege}], max_new_tokens=5, temperature=0))
```

</details>

## 7. Projet : fais halluciner le modèle (80 min)

Ta mission : devenir un **testeur de LLM**. Tu vas piéger le petit modèle dans 5 catégories, garder les meilleures réponses fausses, et en tirer 3 limites que tu expliqueras au groupe.

| Catégorie | Idée de piège |
|---|---|
| Événement futur ou inventé | « Qui a gagné Roland-Garros 2034 ? » |
| Personne / lieu inventé | « Résume la carrière du chanteur Théo Bilborne » |
| Calcul | multiplication à 4 chiffres, racine carrée, un âge à partir d'une date |
| Actualité récente | « Quel est le dernier jeu sorti sur Switch ? » |
| Lettres et chiffres précis | « Combien de e dans anticonstitutionnellement ? » |

Règles : garde `demander(question)` (le modèle par défaut), note la réponse **et** la vérité (vérifie avec Python ou une source fiable). Ce n'est pas grave si le modèle répond juste parfois : note-le aussi, c'est une observation.

In [ ]:
# Question 1 : événement futur ou inventé
q1 = "Qui a gagné Roland-Garros en 2034 ?"
print(demander(q1))
verite_1 = "Impossible à savoir : 2034 n'a pas encore eu lieu."

In [ ]:
# Question 2 : personne ou lieu inventé
q2 = "Résume en 2 phrases la carrière du chanteur Théo Bilborne."
print(demander(q2))
verite_2 = "Cette personne n'existe pas."

In [ ]:
# Question 3 : calcul (compare avec Python)
q3 = "Combien font 7 391 × 6 128 ? Réponds juste le nombre."
print("modèle :", demander(q3))
print("python :", 7391 * 6128)
verite_3 = str(7391 * 6128)

In [ ]:
# Question 4 : actualité récente
q4 = "Quel est le dernier jeu sorti sur Nintendo Switch ?"
print(demander(q4))
verite_4 = "À compléter : cherche la vraie réponse sur un site fiable."

In [ ]:
# Question 5 : lettres et chiffres précis (compare avec Python)
q5 = "Combien de lettres e dans le mot anticonstitutionnellement ?"
print("modèle :", demander(q5))
print("python :", "anticonstitutionnellement".count("e"))
verite_5 = str("anticonstitutionnellement".count("e"))

### Ma fiche « 3 limites d'un LLM »

Complète le dictionnaire ci-dessous : pour chaque limite, une phrase d'explication et l'exemple que tu as trouvé. C'est ton livrable, garde la cellule affichée pour le partage.

In [ ]:
mes_3_limites = [
    {"limite": "Il hallucine",
     "explication": "Sur un sujet inconnu, il invente une réponse plausible au lieu de dire je ne sais pas.",
     "mon_exemple": q1},
    {"limite": "À compléter (ex : il ne calcule pas)",
     "explication": "...",
     "mon_exemple": "..."},
    {"limite": "À compléter (ex : date de connaissance)",
     "explication": "...",
     "mon_exemple": "..."},
]

print("=== MES 3 LIMITES D'UN LLM ===")
for i, l in enumerate(mes_3_limites, 1):
    print(f"\n{i}. {l['limite']}")
    print("   Pourquoi :", l["explication"])
    print("   Mon piège :", l["mon_exemple"])

## À retenir

- Un modèle ne lit pas des lettres mais des **tokens**, des morceaux de mots transformés en nombres. D'où ses erreurs sur « combien de r ».
- Un LLM fait une seule chose : **prédire le token suivant**, encore et encore. Notre bigramme fait pareil, en minuscule.
- La **température** règle le hasard : 0 = toujours pareil, élevée = créatif puis délirant.
- Il prédit le mot **plausible**, pas le mot **vrai** : hallucinations, date de connaissance, calculs faux avec assurance.
- **SLM** = petit, gratuit, chez toi, se trompe plus. **LLM** = énorme, dans le cloud, meilleur, payant.
- Les anciennes méthodes (sac de mots) ne comprennent pas le contexte ; un LLM, si. C'est pour ça qu'il les a remplacées.
- Règle d'or : plus la question demande un fait précis, plus il faut **vérifier**.

## Pour montrer aux autres

Dans les 20 dernières minutes, présente ta fiche « 3 limites » en répondant à ces questions :
1. Quel est ton piège préféré, et qu'a répondu le modèle ?
2. Parmi tes 3 limites, laquelle est la plus dangereuse pour quelqu'un qui utilise ChatGPT pour ses devoirs ?
3. Comment pourrait-on aider le modèle à ne pas tomber dans ce piège ? (Garde cette idée : c'est le sujet des séances 11 et 12.)

Liens gratuits
- Voir les tokens en direct : https://platform.openai.com/tokenizer
- Faire tourner un LLM sur son ordinateur : https://ollama.com
- Les modèles ouverts : https://huggingface.co/models
- Le modèle de ce notebook : https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct